# Historical Scenario Analysis

This notebook reproduces figures and tables for historical hurricane scenarios.

**Paper Figures:**
- **Fig. 2**: Loss decomposition and institutional stress across scenarios

**Supplementary Information:**
- **SI Table 2**: Scenario-based loss decomposition and institutional stress
- **SI Fig. 3**: Wind loss deviation from mean (county-level attribution)

**Data Requirements:**
- `results/mc_runs/scenario_report_with_uncertainty_*.xlsx` (Monte Carlo scenario results)
- `fl_risk_model/data/florida_empirical_hazard_attribution_p95.csv` (wind/water attribution)

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
from matplotlib.colors import Normalize, LogNorm

# Paths
DATA_DIR = Path('../fl_risk_model/data')
RESULTS_DIR = Path('../results')
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = RESULTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

---
## Fig. 2: Loss Decomposition and Institutional Stress

Two-panel figure showing (a) loss decomposition by coverage type and (b) public institutional burden across historical scenarios.

In [ ]:
# Load scenario results - find the most recent report file
scenario_files = sorted(RESULTS_DIR.glob('mc_runs/scenario_report_with_uncertainty_*.xlsx'), 
                        key=lambda p: p.stat().st_mtime, reverse=True)

if not scenario_files:
    raise FileNotFoundError(
        "No scenario results found. Run the historical scenarios first:\n"
        "  python scripts/run/run_deterministic_scenarios.py"
    )

BASELINE_FILE = scenario_files[0]
print(f"Loading: {BASELINE_FILE.name}")

df_baseline = pd.read_excel(BASELINE_FILE, sheet_name='Mean Values')
print(f"Loaded scenario data with {len(df_baseline)} metrics")

In [ ]:
# Extract scenario data
scenarios = ['Great Miami', 'Andrew', 'Double Great Miami', 'Andrew then Great Miami', 
             'Great Miami then Andrew', 'Lake Okeechobee', 'Irma', 'Double Irma']

scenario_data = {}
for scenario in scenarios:
    if scenario in df_baseline.columns:
        metrics = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
        wind_gross = metrics['Wind loss gross (USD)'] / 1e9
        flood_gross = metrics['Flood loss gross (USD)'] / 1e9
        citizens_gross = metrics['Citizens loss gross (USD)'] / 1e9
        wind_under = metrics['Un/underinsured wind (USD)'] / 1e9
        flood_under = metrics['Un/underinsured flood (USD)'] / 1e9
        
        scenario_data[scenario] = {
            'insured_wind': wind_gross - citizens_gross - wind_under,
            'citizens': citizens_gross,
            'insured_flood': flood_gross - flood_under,
            'wind_under': wind_under,
            'flood_under': flood_under
        }

print(f"Extracted data for {len(scenario_data)} scenarios")

In [ ]:
# Fig. 2: Two-panel figure
fig = plt.figure(figsize=(10, 3))
gs = fig.add_gridspec(1, 2, wspace=0.2)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

scenario_order = ['Great Miami', 'Andrew', 'Lake Okeechobee', 'Irma', 
                  'Great Miami then Andrew', 'Double Great Miami', 'Double Irma']

# ===== PANEL A: LOSS DECOMPOSITION =====
insured_wind_vals = [scenario_data[s]['insured_wind'] for s in scenario_order]
citizens_vals = [scenario_data[s]['citizens'] for s in scenario_order]
insured_flood_vals = [scenario_data[s]['insured_flood'] for s in scenario_order]
wind_under_vals = [scenario_data[s]['wind_under'] for s in scenario_order]
flood_under_vals = [scenario_data[s]['flood_under'] for s in scenario_order]

x_pos = np.arange(len(scenario_order))
width = 0.3

# Stacked bars
p1 = ax1.bar(x_pos, insured_wind_vals, width, label='Insured Wind (Private)', color='#E74C3C')
p2 = ax1.bar(x_pos, citizens_vals, width, bottom=insured_wind_vals, label='Citizens Wind', color='#F39C12')
bottom2 = np.array(insured_wind_vals) + np.array(citizens_vals)
p3 = ax1.bar(x_pos, insured_flood_vals, width, bottom=bottom2, label='Insured Flood (NFIP)', color='#3498DB')
bottom3 = bottom2 + np.array(insured_flood_vals)
p4 = ax1.bar(x_pos, wind_under_vals, width, bottom=bottom3, label='Un/Underinsured Wind', color='#95A5A6')
bottom4 = bottom3 + np.array(wind_under_vals)
p5 = ax1.bar(x_pos, flood_under_vals, width, bottom=bottom4, label='Un/Underinsured Flood', color='#BDC3C7')

# Total labels
totals = bottom4 + np.array(flood_under_vals)
for i, total in enumerate(totals):
    ax1.text(i, total + 5, f'${total:.0f}B', ha='center', va='bottom', fontweight='bold', fontsize=9)

ax1.set_ylabel('Loss (Billion USD)', fontsize=11, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(scenario_order, rotation=45, ha='right', fontsize=9)
ax1.legend(loc='upper left', frameon=True, fontsize=8, reverse=True)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.grid(False)
ax1.text(-0.1, 1.05, 'a', transform=ax1.transAxes, fontsize=12, fontweight='bold', va='top')

# ===== PANEL B: INSTITUTIONAL STRESS =====
institutional_data = {}
for scenario in scenario_order:
    if scenario in df_baseline.columns:
        metrics = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
        institutional_data[scenario] = {
            'fhcf_shortfall': metrics.get('FHCF shortfall (USD)', 0) / 1e9,
            'figa_residual': metrics.get('FIGA residual (USD)', 0) / 1e9,
            'citizens_deficit': metrics.get('Citizens residual (USD)', 0) / 1e9,
            'nfip_borrowed': metrics.get('NFIP Treasury borrowing (USD)', 0) / 1e9
        }

fhcf_vals = np.array([institutional_data[s]['fhcf_shortfall'] for s in scenario_order])
figa_vals = np.array([institutional_data[s]['figa_residual'] for s in scenario_order])
citizens_vals_inst = np.array([institutional_data[s]['citizens_deficit'] for s in scenario_order])
nfip_vals = np.array([institutional_data[s]['nfip_borrowed'] for s in scenario_order])
total_public = fhcf_vals + figa_vals + citizens_vals_inst + nfip_vals

bars1 = ax2.bar(x_pos, fhcf_vals, width, label='FHCF Shortfall', color='#DCB7EB')
bars2 = ax2.bar(x_pos, figa_vals, width, bottom=fhcf_vals, label='FIGA Residual', color='#8E44AD')
bars3 = ax2.bar(x_pos, citizens_vals_inst, width, bottom=fhcf_vals + figa_vals, label='Citizens Deficit', color='#F39C12')
bars4 = ax2.bar(x_pos, nfip_vals, width, bottom=fhcf_vals + figa_vals + citizens_vals_inst, label='NFIP Borrowing', color='#3498DB')

for i, total_val in enumerate(total_public):
    if total_val > 0:
        ax2.text(i, total_val + 0.3, f'${total_val:.1f}B', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2.set_ylabel('Public Burden (Billion USD)', fontsize=11, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(scenario_order, rotation=45, ha='right', fontsize=9)
ax2.legend(loc='upper left', frameon=True, fontsize=8, reverse=True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(False)
ax2.text(-0.1, 1.05, 'b', transform=ax2.transAxes, fontsize=12, fontweight='bold', va='top')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_loss_institutional_stress.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR / 'fig_loss_institutional_stress.png'}")

---
## SI Table 2: Scenario-Based Loss Decomposition

Summary table of loss decomposition and institutional stress across all scenarios.

In [ ]:
# Create summary table
summary_data = []
for scenario in scenario_order:
    if scenario in df_baseline.columns:
        metrics = dict(zip(df_baseline['Metric'], df_baseline[scenario]))
        summary_data.append({
            'Scenario': scenario,
            'Total Loss ($B)': metrics.get('Total loss (USD)', 0) / 1e9,
            'Wind Insured ($B)': scenario_data[scenario]['insured_wind'],
            'Citizens ($B)': scenario_data[scenario]['citizens'],
            'Flood Insured ($B)': scenario_data[scenario]['insured_flood'],
            'Wind Uninsured ($B)': scenario_data[scenario]['wind_under'],
            'Flood Uninsured ($B)': scenario_data[scenario]['flood_under'],
            'FHCF Shortfall ($B)': institutional_data[scenario]['fhcf_shortfall'],
            'FIGA Residual ($B)': institutional_data[scenario]['figa_residual'],
            'Citizens Deficit ($B)': institutional_data[scenario]['citizens_deficit'],
            'NFIP Borrowing ($B)': institutional_data[scenario]['nfip_borrowed'],
            'Private Defaults': metrics.get('Private defaults post-group (#)', 0)
        })

df_summary = pd.DataFrame(summary_data)
df_summary.to_csv(TABLES_DIR / 'si_table2_scenario_summary.csv', index=False)
print("SI Table 2: Scenario Summary")
print(df_summary.round(1).to_string(index=False))

---
## SI Fig. 3: Wind Loss Deviation from Mean

County-level deviation of wind loss share from the statewide mean, based on empirical attribution data.

In [ ]:
# Helper function for choropleth
def plot_choropleth(gdf, value_col, title, ax, cmap='cividis', logscale=True,
                   edgecolor='white', linewidth=0.6, missing_color='#f0f0f0'):
    """Plot a choropleth map on the given axes."""
    gdf.plot(ax=ax, color=missing_color, linewidth=0, zorder=0)
    values = gdf[value_col].to_numpy(copy=False)
    
    if logscale:
        positive = values[np.isfinite(values) & (values > 0)]
        vmin = positive.min() if positive.size else 1
        vmax = positive.max() if positive.size else 1
        norm = LogNorm(vmin=vmin, vmax=vmax)
    else:
        finite = values[np.isfinite(values)]
        vmin = np.nanmin(finite) if finite.size else 0
        vmax = np.nanmax(finite) if finite.size else 1
        norm = Normalize(vmin=vmin, vmax=vmax)
    
    gdf.dropna(subset=[value_col]).plot(
        ax=ax, column=value_col, cmap=cmap, norm=norm,
        edgecolor=edgecolor, linewidth=linewidth, zorder=1
    )
    gdf.boundary.plot(ax=ax, color=edgecolor, linewidth=linewidth, zorder=2)
    ax.set_axis_off()
    ax.set_title(title, fontsize=10)
    return norm

In [ ]:
# Load Florida counties
county_fp = DATA_DIR / 'US_counties'
counties = gpd.read_file(county_fp)
fl_counties = counties[counties['STATEFP'] == '12'].copy()
print(f"Loaded {len(fl_counties)} Florida counties")

# Load empirical wind/water attribution
attribution = pd.read_csv(DATA_DIR / 'florida_empirical_hazard_attribution_p95.csv')
attribution['countyfp'] = attribution['admin2_name'].map(
    fl_counties.set_index('NAME')['COUNTYFP'].to_dict()
)
attribution['countyfp'] = attribution['countyfp'].astype(str).str.zfill(3)
print(f"Mean wind share: {attribution['wind_share'].mean():.1f}%")

In [ ]:
# SI Fig. 3: Wind loss deviation map
# Prepare geodataframe
gdf_present = fl_counties[['COUNTYFP', 'NAME', 'geometry']].copy()
gdf_present['countyfp'] = gdf_present['COUNTYFP'].astype(str).str.zfill(3)

# Calculate deviations from mean
empirical_mean_wind = attribution['wind_share'].mean()
attribution_with_dev = attribution.copy()
attribution_with_dev['wind_dev_from_mean'] = attribution_with_dev['wind_share'] - empirical_mean_wind

gdf_present = gdf_present.merge(
    attribution_with_dev[['countyfp', 'wind_dev_from_mean']],
    on='countyfp', how='left'
)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(3.5, 3.5))

norm1 = plot_choropleth(
    gdf_present, 'wind_dev_from_mean', '',
    ax, cmap='RdBu_r', logscale=False
)

# Colorbar
sm1 = plt.cm.ScalarMappable(cmap='RdBu_r', norm=norm1)
sm1.set_array([])
cbar_ax1 = fig.add_axes([0.98, 0.2, 0.03, 0.65])
cbar1 = fig.colorbar(sm1, cax=cbar_ax1)
cbar1.set_label('Wind loss deviation from mean (%)', fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'wind_loss_deviation.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES_DIR / 'wind_loss_deviation.png'}")